# Digital Witness — Full Pipeline



In [10]:
# ============================================================
# CELL 1 — Install Dependencies + Detect Environment
# ============================================================
# Handles: Windows CPU, Linux CPU, CUDA 11.x, CUDA 12.x
# No manual intervention needed — just run this cell.
# ============================================================
import subprocess, sys, re

def _run(cmd):
    subprocess.check_call(cmd, stdout=subprocess.DEVNULL,
                          stderr=subprocess.DEVNULL)

# ---- Detect if we're on Colab ----
try:
    import google.colab
    ON_COLAB = True
except ImportError:
    ON_COLAB = False

# ---- Detect CUDA availability ----
def _detect_cuda_major():
    try:
        out = subprocess.check_output(
            ['nvidia-smi'], text=True, stderr=subprocess.DEVNULL)
        m = re.search(r'CUDA Version:\s*(\d+)', out)
        return int(m.group(1)) if m else 0
    except Exception:
        return 0

cuda_major = _detect_cuda_major()

# ---- Install/verify PyTorch with correct backend ----
try:
    import torch
    has_cuda = torch.cuda.is_available()
    print(f'PyTorch {torch.__version__} already installed. CUDA available: {has_cuda}')
    if not has_cuda and cuda_major >= 11:
        print('  NOTE: GPU detected but PyTorch has no CUDA — reinstalling...')
        _reinstall = True
    else:
        _reinstall = False
except ImportError:
    _reinstall = True

if _reinstall:
    if cuda_major >= 12:
        idx = 'https://download.pytorch.org/whl/cu121'
        print(f'CUDA {cuda_major}.x detected → installing PyTorch cu121...')
    elif cuda_major == 11:
        idx = 'https://download.pytorch.org/whl/cu118'
        print(f'CUDA {cuda_major}.x detected → installing PyTorch cu118...')
    else:
        idx = None
        print('No CUDA GPU → installing CPU-only PyTorch.')

    if idx:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install',
            'torch', 'torchvision', '--index-url', idx, '--quiet'])
    else:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install',
            'torch', 'torchvision', '--quiet'])

# ---- Install remaining packages ----
pkgs = [
    'opencv-python-headless' if ON_COLAB else 'opencv-python',
    'numpy', 'pandas', 'matplotlib', 'scikit-learn',
    'Pillow', 'tqdm', 'pyyaml',
    'ultralytics>=8.3.0', 'lapx>=0.5.2',
]
subprocess.check_call([sys.executable, '-m', 'pip', 'install',
                       *pkgs, '--quiet'])

# ---- Final verification ----
import torch, platform
print()
print('=' * 55)
print('  ENVIRONMENT SUMMARY')
print('=' * 55)
print(f'  OS          : {platform.system()} {platform.release()}')
print(f'  Python      : {sys.version.split()[0]}')
print(f'  PyTorch     : {torch.__version__}')
print(f'  CUDA avail  : {torch.cuda.is_available()}', end='')
if torch.cuda.is_available():
    print(f' — {torch.cuda.get_device_name(0)}')
    print(f'  VRAM        : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print()
    print()
    print('  Running on CPU. Training will be slow.')
    print('  For real training use a CUDA GPU machine.')
    print('  Quick smoke-test (2 epochs) works fine on CPU.')
print('=' * 55)


PyTorch 2.10.0+cpu already installed. CUDA available: False

  ENVIRONMENT SUMMARY
  OS          : Windows 10
  Python      : 3.12.2
  PyTorch     : 2.10.0+cpu
  CUDA avail  : False

  Running on CPU. Training will be slow.
  For real training use a CUDA GPU machine.
  Quick smoke-test (2 epochs) works fine on CPU.


In [11]:
# ============================================================
# CELL 2 — Configuration
# ============================================================
# UPDATE the paths at the top of this cell for your machine.
# Everything else is auto-configured.
# ============================================================
import torch, os, yaml, json, platform
from pathlib import Path
from datetime import datetime

# ---- Detect environment ----
try:
    import google.colab
    ON_COLAB = True
    from google.colab import drive
    drive.mount('/content/drive')
    GDRIVE     = Path('/content/drive/MyDrive')
    ROOT       = GDRIVE / 'DigitalWitness'
    # ── Dataset layout on Google Drive ─────────────────────────────────────────
    # Expected structure (create these folders and place your videos inside):
    #
    #   MyDrive/DigitalWitness/Dataset/
    #       normal/          ← normal videos from:
    #                           • kipshidze/shoplifting-video-dataset (Normal/ folder)
    #                           • UCF-Crime Normal_Videos_event/ folder
    #       shoplifting/     ← shoplifting videos from:
    #                           • kipshidze/shoplifting-video-dataset (Shoplifting/ folder)
    #                           • UCF-Crime Shoplifting/ folder
    #
    #   MyDrive/DigitalWitness/data/dataset/   ← Roboflow YOLO export (unzipped)
    #       data.yaml
    #       train/images/, train/labels/
    #       valid/images/, valid/labels/
    #       test/images/,  test/labels/
    # ────────────────────────────────────────────────────────────────────────────
    VIDEO_ROOT = GDRIVE / 'DigitalWitness' / 'Dataset'
    FRAMES_DIR = Path('/content/frames')      # local SSD — fast I/O on Colab
except ImportError:
    ON_COLAB   = False
    # ── LOCAL MACHINE — update these two lines ──────────────
    # ROOT = Path('D:/Santosh/Project_DigitalWitness')
    ROOT = Path('C:/Users/MSI/Music/Project_DigitalWitness')
    # VIDEO_ROOT should contain two subfolders: normal/ and shoplifting/
    # Combine clips from both Kaggle datasets into these two folders:
    #   normal/      → kipshidze Normal/ clips + UCF-Crime Normal_Videos_event/ clips
    #   shoplifting/ → kipshidze Shoplifting/ clips + UCF-Crime Shoplifting/ clips
    # VIDEO_ROOT = Path('D:/Santosh/Dataset')
    VIDEO_ROOT = Path('C:/Users/MSI/Music/Dataset')
    # ────────────────────────────────────────────────────────
    FRAMES_DIR = ROOT / 'frames'

DATASET_FOLDER = ROOT / 'data' / 'dataset'
MODELS_DIR     = ROOT / 'models'
OUTPUTS_DIR    = ROOT / 'outputs'
CASE_DIR       = OUTPUTS_DIR / 'cases'

for d in [MODELS_DIR, OUTPUTS_DIR, CASE_DIR, FRAMES_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# ---- Device selection ----
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# ---- DataLoader workers (Windows/Jupyter = 0 to avoid deadlock) ----
if platform.system() == 'Windows' or not torch.cuda.is_available():
    NUM_WORKERS = 0
else:
    NUM_WORKERS = 2
PIN_MEMORY = torch.cuda.is_available()

print(f'Device       : {device}')
print(f'Workers      : {NUM_WORKERS}  (0 = safe on Windows/CPU)')
print(f'Pin memory   : {PIN_MEMORY}')
print(f'Root         : {ROOT}')
print(f'Video root   : {VIDEO_ROOT}  (exists: {VIDEO_ROOT.exists()})')
print(f'Dataset      : {DATASET_FOLDER}  (exists: {DATASET_FOLDER.exists()})')

# ---- Patch data.yaml with absolute paths ----
DATASET_YAML = str(DATASET_FOLDER / 'data.yaml')
if Path(DATASET_YAML).exists():
    with open(DATASET_YAML) as f:
        _y = yaml.safe_load(f)
    _y['train'] = str(DATASET_FOLDER / 'train' / 'images')
    _y['val']   = str(DATASET_FOLDER / 'valid' / 'images')
    _y['test']  = str(DATASET_FOLDER / 'test'  / 'images')
    _y.pop('path', None)
    with open(DATASET_YAML, 'w') as f:
        yaml.dump(_y, f, sort_keys=False, allow_unicode=True)
    print()
    print('data.yaml patched:')
    for split in ('train', 'val'):
        p = _y[split]
        ok = 'OK' if Path(p).exists() else 'WARNING: not found'
        print(f'  {split} → {p}  [{ok}]')
else:
    print(f'\nWARNING: data.yaml not found at {DATASET_YAML}')

# ---- YOLO 24-class definitions ----
YOLO_CLASSES = [
    'backpack-or-handbag',                                         # 0
    'carrying-item',                                               # 1
    'empty-basket',                                                # 2
    'empty-shopping-bags',                                         # 3
    'empty-trolly',                                                # 4
    'filled-basket',                                               # 5
    'filled-shopping-bags',                                        # 6
    'filled-trolly',                                               # 7
    'occupied-checkout-counter',                                   # 8
    'person',                                                      # 9
    'person with carrying item-s- and shopping bag-s-',            # 10
    'person-with-backpack-handbag',                                # 11
    'person-with-backpack-handbag-and-carrying-item-s-',           # 12
    'person-with-backpack-handbag-and-shopping-bag-s-',            # 13
    'person-with-basket-trolly-and-backpack-handbag',              # 14
    'person-with-basket-trolly-and-carrying-item',                 # 15
    'person-with-basket-trolly-and-shopping-bag',                  # 16
    'person-with-basket-trolly-shopping-bag-s-and-backpack-handbag', # 17
    'person-with-carrying-item',                                   # 18
    'person-with-empty-basket-trolly',                             # 19
    'person-with-empty-shopping-bag-s-',                           # 20
    'person-with-filled-basket-trolly',                            # 21
    'person-with-filled-shopping-bag-s-',                          # 22
    'vacant-checkout-counter',                                     # 23
]

PERSON_CLASS_IDS  = {9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22}
PRODUCT_HELD_IDS  = {1, 5, 6, 7, 10, 15, 16, 18, 21, 22}
CONCEALMENT_IDS   = {0, 11, 12, 13, 14, 17}
CHECKOUT_OCCUPIED_ID = 8
CHECKOUT_VACANT_ID   = 23
BEHAVIOR_CLASSES  = ['normal', 'shoplifting']

# ---- Model paths ----
YOLO26_BASE    = str(MODELS_DIR / 'yolo26n.pt')
YOLO26_RETAIL  = str(MODELS_DIR / 'yolo26_retail.pt')
MOBILENET_SAVE = str(MODELS_DIR / 'mobilenet_dw.pt')
MOBILENET_INFO = str(MODELS_DIR / 'mobilenet_dw_info.json')

# ---- BiLSTM model path ----
BILSTM_SAVE = str(MODELS_DIR / 'bilstm_dw.pt')
BILSTM_INFO = str(MODELS_DIR / 'bilstm_dw_info.json')

# ---- BiLSTM hyperparameters ----
BILSTM_SEQ_LEN   = 30    # frames per sequence (1 sec @ 30fps)
BILSTM_HIDDEN    = 256   # hidden units per direction
BILSTM_LAYERS    = 2     # stacked BiLSTM layers
BILSTM_DROPOUT   = 0.3
# ┌──────────────────────────────────────────────────────────┐
# │  EPOCHS_BILSTM: 20 for real GPU training                │
# │                  2 for CPU smoke-test only              │
# └──────────────────────────────────────────────────────────┘
EPOCHS_BILSTM    = 1    # GPU: 20  |  CPU smoke-test: 2
LR_BILSTM        = 5e-4
WEIGHT_DECAY_BILSTM = 1e-4   # L2 regularisation — prevents overfitting

# ---- Inference settings ----
YOLO_CONF  = 0.35
YOLO_IOU   = 0.45
FRAME_SIZE = (224, 224)
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]
QUEUE_SIZE    = 30

# ---- Training hyperparameters ----
# ┌─────────────────────────────────────────────────────┐
# │  SMOKE TEST (CPU / quick check):                   │
# │    EPOCHS_YOLO=2, EPOCHS_MOBILENET=2               │
# │  REAL TRAINING (GPU machine):                      │
# │    EPOCHS_YOLO=50, EPOCHS_MOBILENET=25             │
# └─────────────────────────────────────────────────────┘
EPOCHS_YOLO      = 1
EPOCHS_MOBILENET = 1
BATCH_SIZE       = 32 if torch.cuda.is_available() else 16
LR_MOBILENET     = 1e-4
WEIGHT_DECAY_MOBILENET = 1e-4  # L2 regularisation
PATIENCE         = 5

# ── Class imbalance handling ─────────────────────────────────────────────────
# UCF-Crime has ~1,610 normal videos vs ~290 shoplifting — heavily imbalanced.
# kipshidze dataset is balanced (roughly equal normal/shoplifting clips).
# Combined, normal will outnumber shoplifting significantly.
# WeightedRandomSampler is used in the DataLoaders below to compensate.
# If you add more normal videos, increase MAX_NORMAL_VIDEOS to cap them.
MAX_NORMAL_VIDEOS = 500    # cap normal video count — prevents severe imbalance
                            # set None to use all available normal videos
USE_CLASS_WEIGHTS = True   # apply inverse-frequency weights to CrossEntropyLoss

# ---- Resume training -------------------------------------------------------
# True  → load existing model weights and continue training from where it left off
#          (faster, preserves learned features — recommended when adding new data)
# False → train from scratch, replacing any existing saved model
RESUME_TRAINING = False   # ← SET False for a clean training run
#   True  = load existing weights + optimizer and continue (use only if you
#           deliberately want to extend a converged model with NEW data)
#   False = train from scratch (recommended to avoid LR-reset overfitting)
# ---------------------------------------------------------------------------

# ---- Intent thresholds ----
THRESHOLD_LOW      = 0.30
THRESHOLD_MEDIUM   = 0.50
THRESHOLD_HIGH     = 0.70
THRESHOLD_CRITICAL = 0.85

print()
print(f'YOLO classes     : {len(YOLO_CLASSES)}')
print(f'EPOCHS_YOLO      : {EPOCHS_YOLO}')
print(f'EPOCHS_MOBILENET : {EPOCHS_MOBILENET}')
print(f'EPOCHS_BILSTM    : {EPOCHS_BILSTM}')
print(f'BATCH_SIZE       : {BATCH_SIZE}')
print(f'RESUME_TRAINING  : {RESUME_TRAINING}')
print()
if EPOCHS_YOLO <= 2 or EPOCHS_BILSTM <= 2:
    print('NOTE: Running in SMOKE-TEST mode.')
    print('      Metrics will be near random — set full epoch counts for real training.')


Device       : cpu
Workers      : 0  (0 = safe on Windows/CPU)
Pin memory   : False
Root         : C:\Users\MSI\Music\Project_DigitalWitness
Video root   : C:\Users\MSI\Music\Dataset  (exists: True)
Dataset      : C:\Users\MSI\Music\Project_DigitalWitness\data\dataset  (exists: True)

data.yaml patched:
  train → C:\Users\MSI\Music\Project_DigitalWitness\data\dataset\train\images  [OK]
  val → C:\Users\MSI\Music\Project_DigitalWitness\data\dataset\valid\images  [OK]

YOLO classes     : 24
EPOCHS_YOLO      : 1
EPOCHS_MOBILENET : 1
EPOCHS_BILSTM    : 1
BATCH_SIZE       : 16
RESUME_TRAINING  : False

NOTE: Running in SMOKE-TEST mode.
      Metrics will be near random — set full epoch counts for real training.


In [12]:
# ============================================================
# CELL 3 — YOLO26n Fine-tuning
# ============================================================
# Trains YOLO on your 24-class Roboflow annotation dataset.
# freeze=10 keeps the backbone frozen — only the head learns.
# Output: models/yolo26_retail.pt
# ============================================================
from ultralytics import YOLO
import shutil, torch, yaml

# ── Verify Roboflow dataset classes match the 24-class definition ──────────────
# The Roboflow Shoplifting Annotation dataset (v2) has exactly 24 classes which
# match YOLO_CLASSES defined in Cell 2. This check prints a warning if your
# downloaded data.yaml has different class names (e.g. you used a different version).
if Path(DATASET_YAML).exists():
    with open(DATASET_YAML) as _f:
        _dy = yaml.safe_load(_f)
    _roboflow_classes = _dy.get('names', [])
    if len(_roboflow_classes) != len(YOLO_CLASSES):
        print(f'WARNING: data.yaml has {len(_roboflow_classes)} classes but '
              f'notebook expects {len(YOLO_CLASSES)}.')
        print(f'         If you downloaded a different Roboflow version, update '
              f'YOLO_CLASSES in Cell 2 to match.')
    else:
        print(f'YOLO classes verified: {len(_roboflow_classes)} classes match.')
# ────────────────────────────────────────────────────────────────────────────────

if not Path(YOLO26_BASE).exists():
    raise FileNotFoundError(
        f'Base weights not found: {YOLO26_BASE}\n'
        'Download yolo26n.pt and place it in models/.'
    )
if not Path(DATASET_YAML).exists():
    raise FileNotFoundError(f'data.yaml not found: {DATASET_YAML}')

# ---- Resume or start fresh ----
if RESUME_TRAINING and Path(YOLO26_RETAIL).exists():
    _yolo_start = YOLO26_RETAIL
    print(f'Resuming YOLO from fine-tuned weights : {YOLO26_RETAIL}')
else:
    _yolo_start = YOLO26_BASE
    print(f'Training YOLO from base weights       : {YOLO26_BASE}')

print(f'Dataset YAML : {DATASET_YAML}')
print(f'Device       : {device}  ({EPOCHS_YOLO} epochs, batch={BATCH_SIZE})')
print()

yolo = YOLO(_yolo_start)

results = yolo.train(
    data     = DATASET_YAML,
    epochs   = EPOCHS_YOLO,
    imgsz    = 640,
    batch    = BATCH_SIZE,
    freeze   = 10,
    project  = str(ROOT / 'runs'),
    name     = 'yolo26_retail',
    save     = True,
    patience = 5,
    plots    = True,
    device   = 0 if torch.cuda.is_available() else 'cpu',
    workers  = NUM_WORKERS,
    verbose  = True,
)

# Copy best weights regardless of auto-incremented folder name
weights_dir = Path(results.save_dir) / 'weights'
src = (weights_dir / 'best.pt') if (weights_dir / 'best.pt').exists() \
      else (weights_dir / 'last.pt') if (weights_dir / 'last.pt').exists() \
      else None
if src is None:
    raise FileNotFoundError(f'No weights in {weights_dir}')

shutil.copy(str(src), YOLO26_RETAIL)
print(f'\nFine-tuned YOLO saved → {YOLO26_RETAIL}  (from {src.name})')

# Print metrics (handle Ultralytics key variations)
try:
    m = results.results_dict
    keys = list(m.keys())
    map50 = m.get('metrics/mAP50(B)', m.get('metrics/mAP_0.5', 0))
    map95 = m.get('metrics/mAP50-95(B)', m.get('metrics/mAP_0.5:0.95', 0))
    prec  = m.get('metrics/precision(B)', m.get('metrics/precision', 0))
    rec   = m.get('metrics/recall(B)', m.get('metrics/recall', 0))
    print(f'mAP50     : {map50:.3f}')
    print(f'mAP50-95  : {map95:.3f}')
    print(f'Precision : {prec:.3f}')
    print(f'Recall    : {rec:.3f}')
    if EPOCHS_YOLO < 5:
        print(f'\n[Expected: metrics ~0 at {EPOCHS_YOLO} epochs — set EPOCHS_YOLO=50 for real training]')
except Exception as e:
    print(f'Metrics parse skipped: {e}')


YOLO classes verified: 24 classes match.
Training YOLO from base weights       : C:\Users\MSI\Music\Project_DigitalWitness\models\yolo26n.pt
Dataset YAML : C:\Users\MSI\Music\Project_DigitalWitness\data\dataset\data.yaml
Device       : cpu  (1 epochs, batch=16)

New https://pypi.org/project/ultralytics/8.4.21 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.11  Python-3.12.2 torch-2.10.0+cpu CPU (11th Gen Intel Core i5-1155G7 @ 2.50GHz)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=C:\Users\MSI\Music\Project_DigitalWitness\data\dataset\data.yaml, degrees=0.0, deterministic=True, device=cpu, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=1, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=t

## YOLO v2 Fine-Tuning — Domain-Adaptive Training

**Problem:** `yolo26_retail.pt` was trained only on static in-store shelf images
(Roboflow shoplifting-annotation dataset, 694 frames, 24 classes). It detects
**zero objects** on UCF-Crime / Kipshidze surveillance footage — a classic
*domain shift* problem.

**Solution (Option 1):** Use `yolo26n.pt` (COCO base) to auto-label a sample
of behaviour frames with `person` bounding boxes, merge those labels with the
existing retail annotations, then fine-tune a new model on the combined dataset.

Run cells **3b → 3c → 3d** in order, then optionally run **Cell 3e** (Option 3:
re-train MobileNetV2 on tighter person crops from the new YOLO).

In [ ]:
# ============================================================
# CELL 3b — Auto-label behaviour frames with yolo26n.pt
# ============================================================
# Uses the COCO base model to generate person bounding boxes for
# a random sample of surveillance frames, saving them in YOLO label
# format (class 9 = 'person' in the 24-class retail schema).
# Output: data/dataset_v2/train/  (images + labels)
# ============================================================
import shutil, random
from pathlib import Path
from ultralytics import YOLO
from tqdm import tqdm

# ── Config ────────────────────────────────────────────────────────────────────
SAMPLE_SIZE   = 2000   # frames to auto-label (increase to 5000+ on GPU)
CONF_THRESH   = 0.30   # minimum YOLO confidence to accept a person box
PERSON_CLS    = 9      # class index for 'person' in the 24-class schema

V2_TRAIN_IMG  = ROOT / 'data' / 'dataset_v2' / 'train' / 'images'
V2_TRAIN_LBL  = ROOT / 'data' / 'dataset_v2' / 'train' / 'labels'
V2_VAL_IMG    = ROOT / 'data' / 'dataset_v2' / 'valid' / 'images'
V2_VAL_LBL    = ROOT / 'data' / 'dataset_v2' / 'valid' / 'labels'

for d in [V2_TRAIN_IMG, V2_TRAIN_LBL, V2_VAL_IMG, V2_VAL_LBL]:
    d.mkdir(parents=True, exist_ok=True)

# ── Collect behaviour frames ───────────────────────────────────────────────────
all_frames = (
    list((FRAMES_DIR / 'shoplifting').glob('*.jpg')) +
    list((FRAMES_DIR / 'shoplifting').glob('*.png')) +
    list((FRAMES_DIR / 'normal').glob('*.jpg')) +
    list((FRAMES_DIR / 'normal').glob('*.png'))
)
if not all_frames:
    raise FileNotFoundError(
        f'No frames found in {FRAMES_DIR / "shoplifting"} or '
        f'{FRAMES_DIR / "normal"}. Run frame extraction first.'
    )

random.seed(42)
sample = random.sample(all_frames, min(SAMPLE_SIZE, len(all_frames)))
print(f'Total available frames : {len(all_frames):,}')
print(f'Sampling               : {len(sample):,}')
print(f'Output images dir      : {V2_TRAIN_IMG}')

# ── Load base YOLO model ───────────────────────────────────────────────────────
if not Path(YOLO26_BASE).exists():
    raise FileNotFoundError(f'yolo26n.pt not found at {YOLO26_BASE}')
base_model = YOLO(YOLO26_BASE)
print(f'Loaded base model      : {YOLO26_BASE}')

# ── Auto-label loop ────────────────────────────────────────────────────────────
labeled_count = 0
skipped_count = 0   # frames where YOLO found no person

for img_path in tqdm(sample, desc='Auto-labelling'):
    res  = base_model(img_path, conf=CONF_THRESH, verbose=False)[0]
    lines = []
    if res.boxes is not None:
        for box, cls_id in zip(res.boxes.xyxyn.cpu().numpy(),
                                res.boxes.cls.cpu().numpy()):
            if int(cls_id) == 0:   # class 0 = person in COCO
                cx = (box[0] + box[2]) / 2
                cy = (box[1] + box[3]) / 2
                w  =  box[2] - box[0]
                h  =  box[3] - box[1]
                lines.append(f'{PERSON_CLS} {cx:.6f} {cy:.6f} {w:.6f} {h:.6f}')

    if lines:
        dest_img = V2_TRAIN_IMG / img_path.name
        dest_lbl = V2_TRAIN_LBL / img_path.with_suffix('.txt').name
        shutil.copy(str(img_path), str(dest_img))
        dest_lbl.write_text('\n'.join(lines))
        labeled_count += 1
    else:
        skipped_count += 1

print(f'\nAuto-labelled : {labeled_count:,} frames saved to {V2_TRAIN_IMG}')
print(f'Skipped       : {skipped_count:,} frames (no person detected)')
print('\nNext: run Cell 3c to merge with retail annotations and create data.yaml')


In [ ]:
# ============================================================
# CELL 3c — Merge retail annotations into dataset_v2
# ============================================================
# Copies all retail annotation images+labels (from data/dataset/)
# into data/dataset_v2/, then creates a fresh data.yaml that
# covers all 24 classes.
# ============================================================
import shutil, yaml
from pathlib import Path

SRC_TRAIN_IMG = ROOT / 'data' / 'dataset' / 'train' / 'images'
SRC_TRAIN_LBL = ROOT / 'data' / 'dataset' / 'train' / 'labels'
SRC_VAL_IMG   = ROOT / 'data' / 'dataset' / 'valid' / 'images'
SRC_VAL_LBL   = ROOT / 'data' / 'dataset' / 'valid' / 'labels'

def copy_split(src_img, src_lbl, dst_img, dst_lbl, label=''):
    n = 0
    for img in src_img.glob('*.*'):
        lbl = src_lbl / img.with_suffix('.txt').name
        if not lbl.exists():
            continue
        shutil.copy(str(img), str(dst_img / img.name))
        shutil.copy(str(lbl), str(dst_lbl / lbl.name))
        n += 1
    print(f'  {label}: copied {n} images+labels')
    return n

print('Merging retail annotations into dataset_v2/train/ ...')
n_train_retail = copy_split(SRC_TRAIN_IMG, SRC_TRAIN_LBL,
                             V2_TRAIN_IMG, V2_TRAIN_LBL, 'retail train')

print('Merging retail val annotations into dataset_v2/valid/ ...')
n_val_retail = copy_split(SRC_VAL_IMG, SRC_VAL_LBL,
                           V2_VAL_IMG, V2_VAL_LBL, 'retail val')

auto_count = sum(1 for _ in V2_TRAIN_IMG.glob('*.*')) - n_train_retail
print(f'\nDataset_v2 summary:')
print(f'  Train : {sum(1 for _ in V2_TRAIN_IMG.glob("*.*")):,}  '
      f'({n_train_retail} retail + {max(0, auto_count)} auto-labelled)')
print(f'  Val   : {sum(1 for _ in V2_VAL_IMG.glob("*.*")):,}  (retail only)')

# ── Write data.yaml ────────────────────────────────────────────────────────────
V2_YAML_PATH = ROOT / 'data' / 'dataset_v2' / 'data.yaml'

yaml_content = {
    'train': str(V2_TRAIN_IMG),
    'val'  : str(V2_VAL_IMG),
    'nc'   : len(YOLO_CLASSES),
    'names': YOLO_CLASSES,
}
with open(V2_YAML_PATH, 'w') as f:
    yaml.dump(yaml_content, f, sort_keys=False, allow_unicode=True)

print(f'\ndata.yaml written -> {V2_YAML_PATH}')
print(f'  nc={len(YOLO_CLASSES)} classes')
print('\nNext: run Cell 3d to fine-tune YOLO on the merged dataset.')


In [ ]:
# ============================================================
# CELL 3d — Fine-tune YOLO26n on merged dataset_v2
# ============================================================
# Trains YOLO26n on the combined dataset:
#   Retail annotation frames (24 classes, in-store CCTV)
#   Auto-labelled surveillance frames (class 9 = person)
# Output: models/yolo26_dw_v2.pt
# ============================================================
from ultralytics import YOLO
import shutil, torch
from pathlib import Path

YOLO26_DW_V2  = str(MODELS_DIR / 'yolo26_dw_v2.pt')
V2_YAML_PATH  = ROOT / 'data' / 'dataset_v2' / 'data.yaml'

if not V2_YAML_PATH.exists():
    raise FileNotFoundError('dataset_v2/data.yaml not found. Run Cell 3c first.')

# GPU: use at least 30 epochs; CPU smoke-test uses EPOCHS_YOLO from config
_epochs_v2 = max(EPOCHS_YOLO, 30) if torch.cuda.is_available() else EPOCHS_YOLO

print('Starting YOLO v2 fine-tune')
print(f'  Base weights : {YOLO26_BASE}')
print(f'  Dataset YAML : {V2_YAML_PATH}')
print(f'  Epochs       : {_epochs_v2}  (set EPOCHS_YOLO=50 in Cell 2 for full run)')
print(f'  Device       : {device}')

yolo_v2 = YOLO(YOLO26_BASE)

results_v2 = yolo_v2.train(
    data     = str(V2_YAML_PATH),
    epochs   = _epochs_v2,
    imgsz    = 640,
    batch    = BATCH_SIZE,
    freeze   = 10,
    lr0      = 1e-4,
    project  = str(ROOT / 'runs'),
    name     = 'yolo26_dw_v2',
    save     = True,
    patience = 10,
    plots    = True,
    device   = 0 if torch.cuda.is_available() else 'cpu',
    workers  = NUM_WORKERS,
    verbose  = True,
)

weights_dir = Path(results_v2.save_dir) / 'weights'
src = ((weights_dir / 'best.pt') if (weights_dir / 'best.pt').exists()
       else (weights_dir / 'last.pt') if (weights_dir / 'last.pt').exists()
       else None)
if src is None:
    raise FileNotFoundError(f'No weights found in {weights_dir}')

shutil.copy(str(src), YOLO26_DW_V2)
print(f'\nDomain-adaptive YOLO saved -> {YOLO26_DW_V2}  (from {src.name})')

try:
    m     = results_v2.results_dict
    map50 = m.get('metrics/mAP50(B)', m.get('metrics/mAP_0.5', 0))
    map95 = m.get('metrics/mAP50-95(B)', m.get('metrics/mAP_0.5:0.95', 0))
    prec  = m.get('metrics/precision(B)', m.get('metrics/precision', 0))
    rec   = m.get('metrics/recall(B)', m.get('metrics/recall', 0))
    print(f'mAP50     : {map50:.3f}')
    print(f'mAP50-95  : {map95:.3f}')
    print(f'Precision : {prec:.3f}')
    print(f'Recall    : {rec:.3f}')
except Exception as e:
    print(f'Metrics parse skipped: {e}')

print('\nNext: run Cell 3e to update app.py to use yolo26_dw_v2.pt')


In [ ]:
# ============================================================
# CELL 3e — Update app.py to use yolo26_dw_v2.pt
# ============================================================
# Patches get_active_yolo_path() in app.py to prefer the new
# domain-adaptive model over the base yolo26n.pt.
# ============================================================
from pathlib import Path

APP_PATH  = ROOT / 'app.py'
NEW_MODEL = 'yolo26_dw_v2.pt'

if not APP_PATH.exists():
    print(f'app.py not found at {APP_PATH} — skipping patch.')
else:
    src = APP_PATH.read_text(encoding='utf-8')

    # Add YOLO26_DW_V2 constant if missing
    if 'YOLO26_DW_V2' not in src:
        src = src.replace(
            'YOLO26_BASE     = MODELS_DIR / "yolo26n.pt"',
            'YOLO26_BASE     = MODELS_DIR / "yolo26n.pt"\n'
            'YOLO26_DW_V2    = MODELS_DIR / "yolo26_dw_v2.pt"  # domain-adaptive'
        )

    # Update get_active_yolo_path() to check DW_V2 first
    OLD_FN = (
        'def get_active_yolo_path():\n'
        '    """Return base yolo26n.pt for video pipeline'
    )
    NEW_FN = (
        'def get_active_yolo_path():\n'
        '    """Priority: yolo26_dw_v2.pt (domain-adaptive)\n'
        '    -> yolo26n.pt (COCO base) -> yolo26_retail.pt (fallback).\n'
        '    """\n'
        '    if (MODELS_DIR / "yolo26_dw_v2.pt").exists():\n'
        '        return str(MODELS_DIR / "yolo26_dw_v2.pt")\n'
    )

    if OLD_FN in src and 'yolo26_dw_v2' not in src:
        src = src.replace(OLD_FN, NEW_FN)
        APP_PATH.write_text(src, encoding='utf-8')
        print('app.py patched: get_active_yolo_path() now prefers yolo26_dw_v2.pt')
    else:
        print('app.py already references yolo26_dw_v2 or pattern not found.')
        print('Manually set YOLO26_DW_V2 = MODELS_DIR / "yolo26_dw_v2.pt" and')
        print('update get_active_yolo_path() to check it first.')

    new_model_path = ROOT / 'models' / NEW_MODEL
    if new_model_path.exists():
        size_mb = new_model_path.stat().st_size / 1e6
        print(f'Model verified: {new_model_path}  ({size_mb:.1f} MB)')
    else:
        print(f'WARNING: {new_model_path} not found yet. Run Cell 3d first.')


## Option 3 — Re-train MobileNetV2 on Person Crops (Optional)

Now that `yolo26_dw_v2.pt` reliably detects persons across surveillance footage,
re-extracting **tighter person crops** and retraining MobileNetV2 on those crops
(instead of full frames) should push frame-classification accuracy above 98%.

**Why this helps:** Full frames include background clutter (shelves, floor, other
customers). Person crops isolate the subject, giving the classifier a cleaner signal.

Run **Cell 3f** to extract crops, then re-run **Cell 4** with `FRAMES_DIR` pointed
at `frames_v2/` to retrain MobileNetV2.

In [ ]:
# ============================================================
# CELL 3f — Extract person crops with yolo26_dw_v2 (Option 3)
# ============================================================
# Runs the new domain-adaptive YOLO on all behaviour frames,
# saves padded person crops to frames_v2/{normal,shoplifting}/.
# Re-run Cell 4 with FRAMES_DIR = ROOT/'frames_v2' to retrain
# MobileNetV2 on the tighter crops.
# ============================================================
import cv2, random
from pathlib import Path
from ultralytics import YOLO
from tqdm import tqdm

FRAMES_V2_DIR   = ROOT / 'frames_v2'
CROP_PAD_FRAC   = 0.10   # pad each crop by 10% of box dimensions
CROP_CONF       = 0.30   # minimum YOLO confidence to keep
MAX_CROPS_CLASS = 15000  # cap per class
MIN_CROP_PX     = 32     # skip tiny boxes

YOLO26_DW_V2_PATH = MODELS_DIR / 'yolo26_dw_v2.pt'
if not YOLO26_DW_V2_PATH.exists():
    raise FileNotFoundError('yolo26_dw_v2.pt not found. Run Cell 3d first.')

dw_model = YOLO(str(YOLO26_DW_V2_PATH))
print(f'Loaded: {YOLO26_DW_V2_PATH}')

for cls_name in ['normal', 'shoplifting']:
    src_dir = FRAMES_DIR / cls_name
    dst_dir = FRAMES_V2_DIR / cls_name
    dst_dir.mkdir(parents=True, exist_ok=True)

    frames = sorted(src_dir.glob('*.jpg')) + sorted(src_dir.glob('*.png'))
    random.seed(42)
    if len(frames) > MAX_CROPS_CLASS:
        frames = random.sample(frames, MAX_CROPS_CLASS)

    crop_idx = 0
    skip_idx = 0

    for img_path in tqdm(frames, desc=f'Cropping {cls_name}'):
        img = cv2.imread(str(img_path))
        if img is None:
            continue
        H, W = img.shape[:2]

        res = dw_model(img_path, conf=CROP_CONF, verbose=False)[0]
        if res.boxes is None:
            skip_idx += 1
            continue

        for box, cls_id in zip(res.boxes.xyxy.cpu().numpy(),
                                res.boxes.cls.cpu().numpy()):
            cname = dw_model.names[int(cls_id)]
            if not (cname == 'person' or cname.startswith('person')):
                continue
            x1, y1, x2, y2 = map(int, box)
            pad_x = int((x2 - x1) * CROP_PAD_FRAC)
            pad_y = int((y2 - y1) * CROP_PAD_FRAC)
            x1c = max(0, x1 - pad_x)
            y1c = max(0, y1 - pad_y)
            x2c = min(W, x2 + pad_x)
            y2c = min(H, y2 + pad_y)
            if (x2c - x1c) < MIN_CROP_PX or (y2c - y1c) < MIN_CROP_PX:
                continue
            crop = img[y1c:y2c, x1c:x2c]
            cv2.imwrite(str(dst_dir / f'{img_path.stem}_p{crop_idx:04d}.jpg'), crop)
            crop_idx += 1

    print(f'  {cls_name}: {crop_idx:,} crops saved  |  {skip_idx:,} frames skipped')

print(f'\nCrops saved to: {FRAMES_V2_DIR}')
print('To retrain MobileNetV2:')
print('  Change FRAMES_DIR to FRAMES_V2_DIR at the top of Cell 4,')
print('  set RESUME_TRAINING=False, then re-run Cell 4.')


In [15]:
# ============================================================
# CELL 4 — MobileNetV2 Training (behaviour classifier)
# ============================================================
# Trains a binary classifier (normal / shoplifting) on frames
# extracted from your behaviour videos.
# ============================================================
import torch.nn as nn, torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from torchvision.models import MobileNet_V2_Weights
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
import cv2, numpy as np, random
from PIL import Image
from tqdm import tqdm

# ---- Step 4a: Frame extraction ----
def extract_frames(video_root, output_dir, classes, fps_target=2):
    """
    Extract frames from behaviour videos at fps_target frames/sec.

    Dataset notes:
      • kipshidze dataset: folders are named 'Normal' and 'Shoplifting' (capital).
        The notebook maps these to 'normal' and 'shoplifting' (lower-case) via
        VIDEO_ROOT structure — just place videos in normal/ and shoplifting/ folders.
      • UCF-Crime: normal videos live in 'Normal_Videos_event/' and shoplifting
        in 'Shoplifting/'. Copy/symlink the relevant files into the two folders above.
      • Class imbalance: UCF-Crime has ~5x more normal than shoplifting videos.
        MAX_NORMAL_VIDEOS (set in Cell 2) caps the normal class to prevent the
        classifier from simply predicting 'normal' every time.
    """
    total = 0
    for cls in classes:
        src = Path(video_root) / cls
        dst = Path(output_dir) / cls
        dst.mkdir(parents=True, exist_ok=True)

        existing = list(dst.glob('*.jpg'))
        if existing:
            print(f'  [{cls}] {len(existing)} frames already exist — skipping')
            total += len(existing)
            continue

        videos = (list(src.glob('*.mp4')) + list(src.glob('*.avi'))
                  + list(src.glob('*.mov')))
        if not videos:
            print(f'  WARNING [{cls}]: no videos found in {src}')
            print(f'           Expected: {src}')
            print(f'           Place normal/ and shoplifting/ folders in VIDEO_ROOT.')
            continue

        # Cap normal videos to prevent severe class imbalance from UCF-Crime
        if cls == 'normal' and MAX_NORMAL_VIDEOS and len(videos) > MAX_NORMAL_VIDEOS:
            import random as _rng
            _rng.seed(42)
            videos = _rng.sample(videos, MAX_NORMAL_VIDEOS)
            print(f'  [{cls}] Capped to {MAX_NORMAL_VIDEOS} videos '
                  f'(from {len(list(src.glob("*.mp4"))) + len(list(src.glob("*.avi")))}) '
                  f'— set MAX_NORMAL_VIDEOS=None to use all')

        print(f'  [{cls}] {len(videos)} video(s) found — extracting...')
        count = 0
        for vid in tqdm(videos, desc=f'  {cls}', leave=False):
            cap      = cv2.VideoCapture(str(vid))
            vid_fps  = cap.get(cv2.CAP_PROP_FPS) or 25
            interval = max(1, int(vid_fps / fps_target))
            fi = 0
            while True:
                ret, frame = cap.read()
                if not ret: break
                if fi % interval == 0:
                    out = dst / f'{vid.stem}_f{fi:06d}.jpg'
                    cv2.imwrite(str(out), cv2.resize(frame, FRAME_SIZE))
                    count += 1
                fi += 1
            cap.release()
        print(f'  [{cls}] {count} frames extracted')
        total += count
    return total

print('Extracting frames from behaviour videos...')
n_frames = extract_frames(VIDEO_ROOT, FRAMES_DIR, BEHAVIOR_CLASSES, fps_target=2)
print(f'Total frames: {n_frames}')
for cls in BEHAVIOR_CLASSES:
    k = len(list((Path(FRAMES_DIR) / cls).glob('*.jpg')))
    print(f'  {cls}: {k}')

# ---- Validate frame counts before proceeding ----
all_samples = []
for i, cls in enumerate(BEHAVIOR_CLASSES):
    for p in (Path(FRAMES_DIR) / cls).glob('*.jpg'):
        all_samples.append((str(p), i))

if len(all_samples) < 10:
    raise RuntimeError(
        f'Only {len(all_samples)} frames found — need at least 10 to train.\n'
        f'Check that VIDEO_ROOT ({VIDEO_ROOT}) contains\n'
        f'  normal/     (with .mp4/.avi/.mov files)\n'
        f'  shoplifting/ (with .mp4/.avi/.mov files)\n'
    )

random.shuffle(all_samples)
labels_all = [l for _, l in all_samples]
train_s, val_s = train_test_split(all_samples, test_size=0.2,
                                   stratify=labels_all, random_state=42)
print(f'\nTrain: {len(train_s)}  |  Val: {len(val_s)}')

# ---- Step 4b: DataLoaders ----
# BUG FIX: num_workers=NUM_WORKERS (0 on Windows — avoids DataLoader deadlock)
train_tf = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])
val_tf = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

class FrameDataset(Dataset):
    def __init__(self, samples, transform=None):
        self.samples   = samples
        self.transform = transform
    def __len__(self): return len(self.samples)
    def __getitem__(self, idx):
        path, label = self.samples[idx]
        img = Image.open(path).convert('RGB')
        if self.transform: img = self.transform(img)
        return img, label

# Count samples per class (used by both sampler and loss weights below)
if USE_CLASS_WEIGHTS:
    class_counts = [0] * len(BEHAVIOR_CLASSES)
    for _, label in train_s:
        class_counts[label] += 1

# WeightedRandomSampler: oversample shoplifting to balance each mini-batch
if USE_CLASS_WEIGHTS:
    sample_weights = [1.0 / max(class_counts[label], 1) for _, label in train_s]
    sampler = torch.utils.data.WeightedRandomSampler(
        weights=sample_weights, num_samples=len(train_s), replacement=True)
    train_dl = DataLoader(FrameDataset(train_s, train_tf),
                          batch_size=BATCH_SIZE, sampler=sampler,
                          num_workers=NUM_WORKERS,
                          pin_memory=PIN_MEMORY)
    print(f'Using WeightedRandomSampler — shoplifting frames oversampled in each batch')
else:
    train_dl = DataLoader(FrameDataset(train_s, train_tf),
                          batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=NUM_WORKERS,
                          pin_memory=PIN_MEMORY)
val_dl   = DataLoader(FrameDataset(val_s, val_tf),
                      batch_size=BATCH_SIZE, shuffle=False,
                      num_workers=NUM_WORKERS,
                      pin_memory=PIN_MEMORY)
print(f'DataLoaders ready. num_workers={NUM_WORKERS}')

# ---- Step 4c: Build MobileNetV2 ----
# Approach mirrors the ResNet50 reference repos (Rajpurkar et al. 2020):
# freeze backbone → unfreeze last few blocks → train custom head
base = models.mobilenet_v2(weights=MobileNet_V2_Weights.DEFAULT)
for p in base.parameters(): p.requires_grad = False        # freeze all
for layer in base.features[16:]:                           # unfreeze last 3 blocks
    for p in layer.parameters(): p.requires_grad = True
base.classifier = nn.Sequential(
    nn.Dropout(0.4),
    nn.Linear(1280, 512),
    nn.ReLU(),
    nn.Dropout(0.3),
    nn.Linear(512, len(BEHAVIOR_CLASSES)),
)
mobilenet = base.to(device)
trainable = sum(p.numel() for p in mobilenet.parameters() if p.requires_grad)
print(f'Trainable params: {trainable:,}')

# ---- Step 4d: Training loop ----
# Class-weighted loss — compensates for normal >> shoplifting imbalance
if USE_CLASS_WEIGHTS:
    # Count samples per class in training set
    class_counts = [0] * len(BEHAVIOR_CLASSES)
    for _, label in train_s:
        class_counts[label] += 1
    total_samples = sum(class_counts)
    # Inverse-frequency weights: minority class (shoplifting) gets higher weight
    cw = [total_samples / (len(BEHAVIOR_CLASSES) * max(c, 1)) for c in class_counts]
    class_weights = torch.FloatTensor(cw).to(device)
    print(f'Class weights: {dict(zip(BEHAVIOR_CLASSES, [f"{w:.2f}" for w in cw]))}')
    criterion = nn.CrossEntropyLoss(weight=class_weights)
else:
    criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(
    filter(lambda p: p.requires_grad, mobilenet.parameters()),
    lr=LR_MOBILENET,
    weight_decay=WEIGHT_DECAY_MOBILENET)   # FIX 3: L2 regularisation
scheduler  = optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=3, factor=0.5)
history    = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}
best_val_acc, no_improve = 0.0, 0

# ---- Resume from existing model (FIX 1: also restore optimizer state) ----
MOBILENET_OPT_SAVE = str(MODELS_DIR / 'mobilenet_optimizer.pt')
if RESUME_TRAINING and Path(MOBILENET_SAVE).exists():
    mobilenet.load_state_dict(torch.load(MOBILENET_SAVE, map_location=device))
    if Path(MOBILENET_INFO).exists():
        with open(MOBILENET_INFO) as f:
            best_val_acc = json.load(f).get('best_val_acc', 0.0)
    # Restore optimizer so LR scheduler continues from where it left off
    if Path(MOBILENET_OPT_SAVE).exists():
        optimizer.load_state_dict(torch.load(MOBILENET_OPT_SAVE, map_location=device))
        print(f'  Optimizer state restored from {MOBILENET_OPT_SAVE}')
    print(f'Resumed MobileNet  val_acc={best_val_acc:.3f} → continuing from {MOBILENET_SAVE}')
else:
    print('Training MobileNet from scratch.')

for epoch in range(1, EPOCHS_MOBILENET + 1):
    # -- train --
    mobilenet.train()
    tl, tc, tt = 0.0, 0, 0
    train_bar = tqdm(train_dl,
                     desc=f'Epoch {epoch}/{EPOCHS_MOBILENET} [train]',
                     unit='batch', leave=False)
    for imgs, lbls in train_bar:
        imgs, lbls = imgs.to(device), lbls.to(device)
        optimizer.zero_grad()
        out  = mobilenet(imgs)               # BUG 2 FIX: single forward pass
        loss = criterion(out, lbls)
        loss.backward()
        optimizer.step()
        tl += loss.item() * len(imgs)
        tc += (out.argmax(1) == lbls).sum().item()   # reuse out
        tt += len(imgs)
        train_bar.set_postfix(loss=f'{tl/tt:.4f}', acc=f'{tc/tt:.3f}')

    # -- validate --
    mobilenet.eval()
    vl, vc, vt = 0.0, 0, 0
    with torch.no_grad():
        for imgs, lbls in tqdm(val_dl,
                               desc=f'Epoch {epoch}/{EPOCHS_MOBILENET} [val]  ',
                               unit='batch', leave=False):
            imgs, lbls = imgs.to(device), lbls.to(device)
            out = mobilenet(imgs)
            vl += criterion(out, lbls).item() * len(imgs)
            vc += (out.argmax(1) == lbls).sum().item()
            vt += len(imgs)

    tl /= tt; vl /= vt; ta = tc / tt; va = vc / vt
    history['train_loss'].append(tl)
    history['val_loss'].append(vl)
    history['train_acc'].append(ta)
    history['val_acc'].append(va)

    current_lr = optimizer.param_groups[0]['lr']
    print(f'Epoch {epoch:3d}/{EPOCHS_MOBILENET}  '
          f'train_loss={tl:.4f}  train_acc={ta:.3f}  '
          f'val_loss={vl:.4f}  val_acc={va:.3f}  '
          f'lr={current_lr:.2e}')
    scheduler.step(vl)

    if va > best_val_acc:
        best_val_acc = va
        torch.save(mobilenet.state_dict(), MOBILENET_SAVE)
        torch.save(optimizer.state_dict(), MOBILENET_OPT_SAVE)  # FIX 1: save optimizer
        no_improve = 0
        print(f'  ✓ Best saved  val_acc={va:.3f}')
    else:
        no_improve += 1
        if no_improve >= PATIENCE:
            print(f'Early stopping at epoch {epoch}')
            break

with open(MOBILENET_INFO, 'w') as f:
    json.dump({'classes': BEHAVIOR_CLASSES, 'best_val_acc': best_val_acc,
               'input_size': list(FRAME_SIZE),
               'mean': IMAGENET_MEAN, 'std': IMAGENET_STD}, f, indent=2)
print(f'\nBest val accuracy : {best_val_acc:.3f}')
print(f'Model saved       → {MOBILENET_SAVE}')


Extracting frames from behaviour videos...
  [normal] 10730 frames already exist — skipping
  [shoplifting] 20572 frames already exist — skipping
Total frames: 31302
  normal: 10730
  shoplifting: 20572

Train: 25041  |  Val: 6261
Using WeightedRandomSampler — shoplifting frames oversampled in each batch
DataLoaders ready. num_workers=0
Trainable params: 1,862,978
Class weights: {'normal': '1.46', 'shoplifting': '0.76'}
Training MobileNet from scratch.


Epoch   1/1  train_loss=0.1118  train_acc=0.942  val_loss=0.0524  val_acc=0.977  lr=1.00e-04
  ✓ Best saved  val_acc=0.977

Best val accuracy : 0.977
Model saved       → C:\Users\MSI\Music\Project_DigitalWitness\models\mobilenet_dw.pt


In [17]:
# ============================================================
# CELL 4b — BiLSTM + Attention Training
# ============================================================
# Takes MobileNetV2 feature sequences and trains a BiLSTM with
# a temporal attention mechanism to classify behaviour.
#
# Pipeline:
#   Sequence of frames  →  MobileNetV2 (frozen, feature extractor)
#   → feature vectors (1280-d each, SEQ_LEN frames)
#   → BiLSTM (bidirectional, 2 layers)
#   → Attention mechanism (learns which frames matter most)
#   → classifier head → normal / shoplifting
#
# The attention weights are saved per-inference and used as
# the XAI temporal explanation in the case file.
# ============================================================
import torch.nn as nn, torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from torchvision.models import MobileNet_V2_Weights
from sklearn.model_selection import train_test_split
import numpy as np, random, json
from pathlib import Path
from tqdm import tqdm

# ── Step 4b-i: Attention module ──────────────────────────────────────────────
class TemporalAttention(nn.Module):
    """
    Additive attention over the BiLSTM output sequence.
    Produces a context vector + attention weights (used for XAI).
    """
    def __init__(self, hidden_size):
        super().__init__()
        self.attn = nn.Linear(hidden_size, 1)

    def forward(self, lstm_out):
        # lstm_out: (batch, seq_len, hidden_size)
        scores  = self.attn(lstm_out).squeeze(-1)          # (batch, seq_len)
        weights = torch.softmax(scores, dim=1)             # (batch, seq_len)
        context = (weights.unsqueeze(-1) * lstm_out).sum(1) # (batch, hidden_size)
        return context, weights


# ── Step 4b-ii: Full CNN-BiLSTM-Attention model ───────────────────────────────
class CNNBiLSTMAttention(nn.Module):
    """
    CNN feature extractor (MobileNetV2, frozen) →
    BiLSTM → Temporal Attention → classifier.

    Input : (batch, seq_len, 3, 224, 224)
    Output: logits (batch, num_classes),
            attention_weights (batch, seq_len)
    """
    def __init__(self, num_classes, hidden=BILSTM_HIDDEN,
                 layers=BILSTM_LAYERS, dropout=BILSTM_DROPOUT):
        super().__init__()

        # CNN backbone — fully frozen (features extracted offline)
        base = models.mobilenet_v2(weights=MobileNet_V2_Weights.DEFAULT)
        self.cnn = nn.Sequential(*list(base.features),
                                  nn.AdaptiveAvgPool2d(1))
        for p in self.cnn.parameters():
            p.requires_grad = False

        feat_dim = 1280   # MobileNetV2 output channels

        self.bilstm = nn.LSTM(
            input_size  = feat_dim,
            hidden_size = hidden,
            num_layers  = layers,
            batch_first = True,
            bidirectional = True,
            dropout = dropout if layers > 1 else 0.0,
        )
        self.attention  = TemporalAttention(hidden * 2)   # *2 for bidirectional
        self.classifier = nn.Sequential(
            nn.LayerNorm(hidden * 2),
            nn.Dropout(dropout),
            nn.Linear(hidden * 2, 128),
            nn.ReLU(),
            nn.Linear(128, num_classes),
        )

    def forward(self, x):
        # x: (batch, seq_len, C, H, W)
        B, T, C, H, W = x.shape
        # Extract CNN features for every frame
        feats = self.cnn(x.view(B * T, C, H, W))          # (B*T, 1280, 1, 1)
        feats = feats.view(B, T, -1)                       # (B, T, 1280)
        # BiLSTM over the sequence
        lstm_out, _ = self.bilstm(feats)                   # (B, T, hidden*2)
        # Attention
        context, attn_w = self.attention(lstm_out)         # (B, hidden*2), (B, T)
        # Classify
        logits = self.classifier(context)                  # (B, num_classes)
        return logits, attn_w


# ── Step 4b-iii: Sequence dataset ────────────────────────────────────────────
class SequenceDataset(Dataset):
    """
    Builds fixed-length frame sequences from the already-extracted frames.
    Frames are loaded, sorted by name (time order), then chunked into
    non-overlapping windows of length SEQ_LEN.
    """
    def __init__(self, frame_root, classes, seq_len, transform=None):
        self.seq_len   = seq_len
        self.transform = transform
        self.sequences = []   # list of (list_of_paths, label_int)

        for label_idx, cls in enumerate(classes):
            cls_dir = Path(frame_root) / cls
            paths   = sorted(cls_dir.glob('*.jpg'))
            # Group consecutive frames by source video (stem before _f)
            from collections import defaultdict
            vid_frames = defaultdict(list)
            for p in paths:
                # filename: videoname_f000000.jpg
                vid_key = '_'.join(p.stem.split('_')[:-1])
                vid_frames[vid_key].append(p)

            for vid_key, vid_paths in vid_frames.items():
                vid_paths = sorted(vid_paths)
                # Slide non-overlapping windows
                for start in range(0, len(vid_paths) - seq_len + 1, seq_len):
                    chunk = vid_paths[start:start + seq_len]
                    if len(chunk) == seq_len:
                        self.sequences.append((chunk, label_idx))

        # FIX 2: shuffle removed from here — was causing data leakage across runs
        # because sequences were shuffled BEFORE train/val split, meaning different
        # runs produced different splits. DataLoader(shuffle=True) handles shuffling.
        # random.shuffle(self.sequences)  ← REMOVED

    def __len__(self):
        return len(self.sequences)

    def __getitem__(self, idx):
        paths, label = self.sequences[idx]
        frames = []
        for p in paths:
            img = Image.open(p).convert('RGB')
            if self.transform:
                img = self.transform(img)
            frames.append(img)
        return torch.stack(frames), label   # (seq_len, C, H, W), int


seq_tf = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

print('Building sequence dataset...')
full_seq_ds = SequenceDataset(FRAMES_DIR, BEHAVIOR_CLASSES,
                               BILSTM_SEQ_LEN, transform=seq_tf)

if len(full_seq_ds) < 4:
    raise RuntimeError(
        f'Only {len(full_seq_ds)} sequences found.\n'
        f'Need at least {BILSTM_SEQ_LEN} consecutive frames per video.\n'
        f'Make sure Cell 4 frame extraction ran successfully first.'
    )

# Train / val split
labels_seq   = [full_seq_ds.sequences[i][1] for i in range(len(full_seq_ds))]
train_idx, val_idx = train_test_split(
    range(len(full_seq_ds)), test_size=0.2,
    stratify=labels_seq, random_state=42)

from torch.utils.data import Subset
# Count sequences per class (used by sampler and loss weights below)
if USE_CLASS_WEIGHTS:
    seq_class_counts = [0] * len(BEHAVIOR_CLASSES)
    for _, label in full_seq_ds.sequences:
        seq_class_counts[label] += 1

# WeightedRandomSampler for BiLSTM — same imbalance fix as MobileNetV2
if USE_CLASS_WEIGHTS:
    seq_sample_weights = [
        1.0 / max(seq_class_counts[full_seq_ds.sequences[i][1]], 1)
        for i in train_idx
    ]
    seq_sampler = torch.utils.data.WeightedRandomSampler(
        weights=seq_sample_weights, num_samples=len(train_idx), replacement=True)
    train_seq_dl = DataLoader(
        Subset(full_seq_ds, train_idx),
        batch_size=8, sampler=seq_sampler,
        num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)
    print('BiLSTM using WeightedRandomSampler')
else:
    train_seq_dl = DataLoader(
        Subset(full_seq_ds, train_idx),
        batch_size=8, shuffle=True,
        num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)
val_seq_dl = DataLoader(
    Subset(full_seq_ds, val_idx),
    batch_size=8, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)

print(f'Sequences  — train: {len(train_idx)}  val: {len(val_idx)}')
print(f'Seq length : {BILSTM_SEQ_LEN} frames')


# ── Step 4b-iv: Build model ───────────────────────────────────────────────────
bilstm_model = CNNBiLSTMAttention(
    num_classes = len(BEHAVIOR_CLASSES),
    hidden  = BILSTM_HIDDEN,
    layers  = BILSTM_LAYERS,
    dropout = BILSTM_DROPOUT,
).to(device)

trainable_bilstm = sum(
    p.numel() for p in bilstm_model.parameters() if p.requires_grad)
print(f'BiLSTM trainable params: {trainable_bilstm:,}')


# ── Step 4b-v: Training loop ─────────────────────────────────────────────────
# Class-weighted loss for BiLSTM — mirrors MobileNetV2 treatment above
if USE_CLASS_WEIGHTS:
    seq_class_counts = [0] * len(BEHAVIOR_CLASSES)
    for _, label in full_seq_ds.sequences:
        seq_class_counts[label] += 1
    seq_total = sum(seq_class_counts)
    seq_cw = [seq_total / (len(BEHAVIOR_CLASSES) * max(c, 1))
              for c in seq_class_counts]
    seq_weights = torch.FloatTensor(seq_cw).to(device)
    print(f'BiLSTM class weights: {dict(zip(BEHAVIOR_CLASSES, [f"{w:.2f}" for w in seq_cw]))}')
    criterion_b = nn.CrossEntropyLoss(weight=seq_weights)
else:
    criterion_b = nn.CrossEntropyLoss()

optimizer_b = optim.Adam(
    filter(lambda p: p.requires_grad, bilstm_model.parameters()),
    lr=LR_BILSTM,
    weight_decay=WEIGHT_DECAY_BILSTM)   # FIX 3: L2 regularisation
scheduler_b = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer_b, patience=3, factor=0.5)

history_b   = {'train_loss': [], 'val_loss': [],
                'train_acc': [],  'val_acc': []}
best_bilstm_acc, no_imp_b = 0.0, 0

# ---- Resume from existing model (FIX 1: also restore optimizer state) ----
BILSTM_OPT_SAVE = str(MODELS_DIR / 'bilstm_optimizer.pt')
if RESUME_TRAINING and Path(BILSTM_SAVE).exists():
    bilstm_model.load_state_dict(torch.load(BILSTM_SAVE, map_location=device))
    if Path(BILSTM_INFO).exists():
        with open(BILSTM_INFO) as f:
            best_bilstm_acc = json.load(f).get('best_val_acc', 0.0)
    # FIX 1: Restore optimizer so LR scheduler continues from where it left off
    # This prevents the LR from resetting to LR_BILSTM and destroying converged weights
    if Path(BILSTM_OPT_SAVE).exists():
        optimizer_b.load_state_dict(torch.load(BILSTM_OPT_SAVE, map_location=device))
        print(f'  Optimizer state restored — LR continues from last checkpoint')
    print(f'Resumed BiLSTM  val_acc={best_bilstm_acc:.3f} → continuing from {BILSTM_SAVE}')
else:
    print('Training BiLSTM from scratch.')

for epoch in range(1, EPOCHS_BILSTM + 1):
    bilstm_model.train()
    tl, tc, tt = 0.0, 0, 0
    for seqs, lbls in tqdm(train_seq_dl,
                            desc=f'BiLSTM Epoch {epoch}/{EPOCHS_BILSTM} [train]',
                            leave=False):
        seqs, lbls = seqs.to(device), lbls.to(device)
        optimizer_b.zero_grad()
        logits, _ = bilstm_model(seqs)
        loss = criterion_b(logits, lbls)
        loss.backward()
        nn.utils.clip_grad_norm_(bilstm_model.parameters(), max_norm=1.0)
        optimizer_b.step()
        tl += loss.item() * len(seqs)
        tc += (logits.argmax(1) == lbls).sum().item()
        tt += len(seqs)

    bilstm_model.eval()
    vl, vc, vt = 0.0, 0, 0
    with torch.no_grad():
        for seqs, lbls in tqdm(val_seq_dl,
                                desc=f'BiLSTM Epoch {epoch}/{EPOCHS_BILSTM} [val]  ',
                                leave=False):
            seqs, lbls = seqs.to(device), lbls.to(device)
            logits, _ = bilstm_model(seqs)
            vl += criterion_b(logits, lbls).item() * len(seqs)
            vc += (logits.argmax(1) == lbls).sum().item()
            vt += len(seqs)

    tl /= tt; vl /= vt; ta = tc / tt; va = vc / vt
    history_b['train_loss'].append(tl)
    history_b['val_loss'].append(vl)
    history_b['train_acc'].append(ta)
    history_b['val_acc'].append(va)

    lr_now = optimizer_b.param_groups[0]['lr']
    print(f'BiLSTM Epoch {epoch:3d}/{EPOCHS_BILSTM}  '
          f'train_loss={tl:.4f}  train_acc={ta:.3f}  '
          f'val_loss={vl:.4f}  val_acc={va:.3f}  lr={lr_now:.2e}')
    scheduler_b.step(vl)

    if va > best_bilstm_acc:
        best_bilstm_acc = va
        torch.save(bilstm_model.state_dict(), BILSTM_SAVE)
        torch.save(optimizer_b.state_dict(), BILSTM_OPT_SAVE)  # FIX 1: save optimizer
        no_imp_b = 0
        print(f'  ✓ Best saved  val_acc={va:.3f}')
    else:
        no_imp_b += 1
        if no_imp_b >= PATIENCE:
            print(f'Early stopping at epoch {epoch}')
            break

with open(BILSTM_INFO, 'w') as f:
    json.dump({'classes': BEHAVIOR_CLASSES,
               'best_val_acc': best_bilstm_acc,
               'seq_len': BILSTM_SEQ_LEN,
               'hidden': BILSTM_HIDDEN,
               'layers': BILSTM_LAYERS}, f, indent=2)
print(f'\nBiLSTM best val accuracy : {best_bilstm_acc:.3f}')
print(f'Model saved              → {BILSTM_SAVE}')
if EPOCHS_BILSTM <= 2:
    print(f'[Expected: metrics ~random at {EPOCHS_BILSTM} epochs '
          f'— set EPOCHS_BILSTM=20 for real training]')


Building sequence dataset...
BiLSTM using WeightedRandomSampler
Sequences  — train: 700  val: 176
Seq length : 30 frames
BiLSTM trainable params: 4,794,243
BiLSTM class weights: {'normal': '1.60', 'shoplifting': '0.73'}
Training BiLSTM from scratch.


BiLSTM Epoch   1/1  train_loss=0.3173  train_acc=0.876  val_loss=0.1208  val_acc=0.972  lr=5.00e-04
  ✓ Best saved  val_acc=0.972

BiLSTM best val accuracy : 0.972
Model saved              → C:\Users\MSI\Music\Project_DigitalWitness\models\bilstm_dw.pt
[Expected: metrics ~random at 1 epochs — set EPOCHS_BILSTM=20 for real training]


In [18]:
# ============================================================
# CELL 5 — Evaluation (MobileNetV2)
# ============================================================
from sklearn.metrics import classification_report, ConfusionMatrixDisplay, confusion_matrix
from tqdm import tqdm
import numpy as np

mobilenet.load_state_dict(torch.load(MOBILENET_SAVE, map_location=device))
mobilenet.eval()

all_preds, all_labels = [], []
with torch.no_grad():
    for imgs, lbls in tqdm(val_dl, desc='Evaluating', unit='batch'):
        preds = mobilenet(imgs.to(device)).argmax(1).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(lbls.numpy())

print('=== MobileNetV2 Classification Report ===')
print(classification_report(all_labels, all_preds, target_names=BEHAVIOR_CLASSES))

cm = confusion_matrix(all_labels, all_preds)
fig, ax = plt.subplots(figsize=(6, 5))
ConfusionMatrixDisplay(cm, display_labels=BEHAVIOR_CLASSES).plot(
    ax=ax, cmap='Blues', colorbar=False)
ax.set_title('Confusion Matrix — Digital Witness')
plt.tight_layout()
plt.savefig(str(OUTPUTS_DIR / 'confusion_matrix.png'), dpi=150)
plt.show()

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(history['train_loss'], label='Train'); ax1.plot(history['val_loss'], label='Val')
ax1.set_title('Loss'); ax1.legend()
ax2.plot(history['train_acc'], label='Train'); ax2.plot(history['val_acc'], label='Val')
ax2.set_title('Accuracy'); ax2.legend()
plt.tight_layout()
plt.savefig(str(OUTPUTS_DIR / 'learning_curve.png'), dpi=150)
plt.show()


Evaluating: 100%|██████████| 392/392 [02:07<00:00,  3.07batch/s]


=== MobileNetV2 Classification Report ===
              precision    recall  f1-score   support

      normal       0.96      0.98      0.97      2146
 shoplifting       0.99      0.98      0.98      4115

    accuracy                           0.98      6261
   macro avg       0.97      0.98      0.97      6261
weighted avg       0.98      0.98      0.98      6261



<Figure size 600x500 with 1 Axes>

<Figure size 1200x400 with 2 Axes>

In [22]:
# ============================================================
# CELL 6 — PersonProductTracker + POS Integration
# ============================================================
import sys
sys.path.insert(0, str(ROOT))

from pos_integration import (
    MockPOSDatabase,
    POSComparator,
    prompt_operator_verification,
    generate_pos_report,
)

class PersonProductTracker:
    """Per-person product state tracked from YOLO class detections."""
    def __init__(self, track_id):
        self.track_id           = track_id
        self.max_products_held  = 0
        self.concealment_frames = 0
        self.checkout_visited   = False
        self.product_frames     = 0
        self.total_frames       = 0

    def update(self, detected_class_ids, checkout_occupied_nearby):
        self.total_frames += 1
        products_now = len(detected_class_ids & PRODUCT_HELD_IDS)
        self.max_products_held = max(self.max_products_held, products_now)
        if products_now > 0:
            self.product_frames += 1
        if detected_class_ids & CONCEALMENT_IDS:
            self.concealment_frames += 1
        if checkout_occupied_nearby:
            self.checkout_visited = True

    @property
    def left_with_goods(self):
        return self.max_products_held > 0 and not self.checkout_visited

    @property
    def concealment_ratio(self):
        return self.concealment_frames / max(1, self.total_frames)

    def to_dict(self):
        return {
            'track_id'          : self.track_id,
            'max_products_held' : self.max_products_held,
            'concealment_frames': self.concealment_frames,
            'concealment_ratio' : round(self.concealment_ratio, 3),
            'checkout_visited'  : self.checkout_visited,
            'left_with_goods'   : self.left_with_goods,
            'total_frames'      : self.total_frames,
        }

# ---- Generate mock POS database ----
POS_DB_PATH = str(OUTPUTS_DIR / 'mock_pos_transactions.json')

pos_db = MockPOSDatabase()
pos_db.generate_sessions(n=20)

# Add a suspicious test session timestamped to ~now
suspicious_txn = pos_db.add_suspicious_session(
    timestamp      = datetime.now(),
    items_billed   = 2,
    items_detected = 5,
)
pos_db.save(POS_DB_PATH)
pos_db.summary()
print()
print(f'Suspicious test session : {suspicious_txn["session_id"]}')
print(f'  Billed: {suspicious_txn["items_billed"]} items  '
      f'(YOLO should detect ~5 in test video)')


Generated 20 mock transactions.
Added suspicious session: 2 billed, 5 detected (mismatch: 3)
Saved 21 transactions → C:\Users\MSI\Music\Project_DigitalWitness\outputs\mock_pos_transactions.json
Transactions  : 21
Total items   : 99
Avg items/txn : 4.7
Time range    : 09:01 – 23:49

Suspicious test session : SES-SUSP-8009
  Billed: 2 items  (YOLO should detect ~5 in test video)


In [23]:
# ============================================================
# CELL 7 — Full Inference Pipeline (CNN-BiLSTM-Attention)
# ============================================================
# YOLO detects persons + products per frame.
# For each tracked person:
#   - Person crops fed through MobileNetV2 (feature extractor)
#   - Feature sequence fed through BiLSTM + Attention
#   - Attention weights saved as XAI temporal explanation
#   - PersonProductTracker updated from YOLO class IDs
# ============================================================
import torch.nn.functional as F
from collections import deque, defaultdict
import cv2, numpy as np
from torchvision import models, transforms
import torch.nn as nn

# Inference transform (same as training)
inf_tf = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize(FRAME_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

def load_mobilenet(model_path):
    """Load MobileNetV2 as standalone classifier (used for Cell 5 eval)."""
    m = models.mobilenet_v2(weights=None)
    m.classifier = nn.Sequential(
        nn.Dropout(0.4), nn.Linear(1280, 512),
        nn.ReLU(), nn.Dropout(0.3),
        nn.Linear(512, len(BEHAVIOR_CLASSES)),
    )
    m.load_state_dict(torch.load(model_path, map_location=device))
    return m.to(device).eval()

def load_bilstm(model_path):
    """Load the CNN-BiLSTM-Attention model for inference."""
    m = CNNBiLSTMAttention(
        num_classes = len(BEHAVIOR_CLASSES),
        hidden  = BILSTM_HIDDEN,
        layers  = BILSTM_LAYERS,
        dropout = BILSTM_DROPOUT,
    )
    m.load_state_dict(torch.load(model_path, map_location=device))
    return m.to(device).eval()

def _extract_cnn_feature(crop_bgr, cnn_backbone):
    """
    Extract 1280-d MobileNetV2 feature vector from a single BGR crop.
    Used to build the per-person feature sequence for the BiLSTM.
    """
    rgb    = cv2.cvtColor(crop_bgr, cv2.COLOR_BGR2RGB)
    tensor = inf_tf(rgb).unsqueeze(0).to(device)     # (1, 3, 224, 224)
    feat   = cnn_backbone(tensor)                     # (1, 1280, 1, 1)
    return feat.view(1280).cpu()                      # (1280,)


def run_inference(video_path,
                  yolo_path      = None,
                  bilstm_path    = None,
                  queue_size     = QUEUE_SIZE,
                  frame_step     = 1):
    """
    Full inference using CNN-BiLSTM-Attention pipeline.

    Per tracked person:
      - CNN extracts 1280-d feature per frame crop
      - Features queued in a sliding window (SEQ_LEN frames)
      - When queue is full, BiLSTM+Attention classifies the sequence
      - Attention weights are stored as XAI explanation

    Returns dict:
        overall_class      — 'normal' or 'shoplifting'
        overall_conf       — 0-1
        behavior_events    — list of timed segments
        person_summaries   — per-person product evidence (POS-ready)
        attention_maps     — {track_id: list of attention weight arrays}
        duration, fps, total_frames
    """
    from ultralytics import YOLO as _YOLO

    yolo_path   = yolo_path   or YOLO26_RETAIL
    bilstm_path = bilstm_path or BILSTM_SAVE

    yolo_model   = _YOLO(yolo_path)
    bilstm_model_inf = load_bilstm(bilstm_path)

    # CNN backbone only — for feature extraction
    _base = models.mobilenet_v2(weights=None)
    cnn_backbone = nn.Sequential(
        *list(_base.features),
        nn.AdaptiveAvgPool2d(1)
    )
    # Load weights from the BiLSTM model's CNN portion
    cnn_state = {k.replace('cnn.', ''): v
                 for k, v in torch.load(bilstm_path, map_location=device).items()
                 if k.startswith('cnn.')}
    cnn_backbone.load_state_dict(cnn_state)
    cnn_backbone = cnn_backbone.to(device).eval()

    cap      = cv2.VideoCapture(str(video_path))
    fps      = cap.get(cv2.CAP_PROP_FPS) or 25
    total_f  = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    duration = total_f / fps

    # Per-person feature queues (sliding window for BiLSTM input)
    person_feat_queues  = defaultdict(lambda: deque(maxlen=BILSTM_SEQ_LEN))
    person_trackers     = {}
    person_attn_maps    = defaultdict(list)   # XAI: attention per inference step

    behavior_events = []
    current_event   = None
    frame_num       = 0

    with torch.no_grad():
        while True:
            ret, frame = cap.read()
            if not ret: break
            frame_num += 1
            if frame_num % frame_step != 0: continue
            timestamp = frame_num / fps

            # YOLO + ByteTrack
            yolo_res = yolo_model.track(
                frame, persist=True,
                conf=YOLO_CONF, iou=YOLO_IOU, verbose=False
            )[0]

            frame_classes = set()
            if yolo_res.boxes is not None and len(yolo_res.boxes):
                for c in yolo_res.boxes.cls.cpu().numpy():
                    frame_classes.add(int(c))
            checkout_occupied = CHECKOUT_OCCUPIED_ID in frame_classes

            frame_pred_label = 'normal'
            frame_pred_conf  = 0.5

            if yolo_res.boxes is not None and len(yolo_res.boxes):
                boxes     = yolo_res.boxes
                cls_ids   = boxes.cls.cpu().numpy().astype(int)
                xyxy      = boxes.xyxy.cpu().numpy().astype(int)
                track_ids = (boxes.id.cpu().numpy().astype(int)
                             if boxes.id is not None
                             else np.arange(len(boxes)))

                person_classes = defaultdict(set)
                person_boxes   = {}
                for i, (cid, tid) in enumerate(zip(cls_ids, track_ids)):
                    if cid in PERSON_CLASS_IDS:
                        person_classes[tid].add(cid)
                        person_boxes[tid] = xyxy[i]

                for tid, cls_set in person_classes.items():
                    if tid not in person_trackers:
                        person_trackers[tid] = PersonProductTracker(tid)
                    person_trackers[tid].update(cls_set, checkout_occupied)

                    x1, y1, x2, y2 = person_boxes[tid]
                    x1, y1 = max(0, x1), max(0, y1)
                    x2 = min(frame.shape[1], x2)
                    y2 = min(frame.shape[0], y2)

                    if x2 > x1 and y2 > y1:
                        crop = frame[y1:y2, x1:x2]
                        # Extract CNN feature for this frame
                        feat = _extract_cnn_feature(crop, cnn_backbone)
                        person_feat_queues[tid].append(feat)

                        # Only classify when we have a full sequence
                        if len(person_feat_queues[tid]) == BILSTM_SEQ_LEN:
                            seq_tensor = torch.stack(
                                list(person_feat_queues[tid])
                            ).unsqueeze(0).to(device)   # (1, SEQ_LEN, 1280)

                            # Run through BiLSTM+Attention
                            # Note: bilstm_model expects image sequences,
                            # but we pass pre-extracted features via the
                            # CNN bypass path below
                            # ── Direct feature-mode forward pass ──────────
                            feats_in = seq_tensor                          # (1, T, 1280)
                            lstm_out, _ = bilstm_model_inf.bilstm(feats_in)
                            context, attn_w = bilstm_model_inf.attention(lstm_out)
                            logits = bilstm_model_inf.classifier(context)  # (1, 2)
                            probs  = F.softmax(logits, dim=1).cpu().numpy()[0]

                            # Store attention weights for XAI
                            person_attn_maps[tid].append(
                                attn_w.squeeze(0).cpu().numpy().tolist())

                            pid = int(probs.argmax())
                            if (BEHAVIOR_CLASSES[pid] == 'shoplifting'
                                    and probs[pid] > frame_pred_conf):
                                frame_pred_label = 'shoplifting'
                                frame_pred_conf  = float(probs[pid])
                            elif frame_pred_label == 'normal':
                                frame_pred_conf  = float(probs[pid])

            # Build behaviour event segments
            if current_event is None:
                current_event = {'behavior_type': frame_pred_label,
                                 'start_time': timestamp, 'end_time': timestamp,
                                 'confidence': frame_pred_conf}
            elif frame_pred_label == current_event['behavior_type']:
                current_event['end_time']   = timestamp
                current_event['confidence'] = max(current_event['confidence'],
                                                   frame_pred_conf)
            else:
                behavior_events.append(current_event)
                current_event = {'behavior_type': frame_pred_label,
                                 'start_time': timestamp, 'end_time': timestamp,
                                 'confidence': frame_pred_conf}

    cap.release()
    if current_event:
        behavior_events.append(current_event)

    shop_w = sum(e['confidence'] * (e['end_time'] - e['start_time'])
                 for e in behavior_events if e['behavior_type'] == 'shoplifting')
    norm_w = sum(e['confidence'] * (e['end_time'] - e['start_time'])
                 for e in behavior_events if e['behavior_type'] == 'normal')
    overall_class = 'shoplifting' if shop_w >= norm_w else 'normal'
    overall_conf  = min(1.0, (shop_w if overall_class == 'shoplifting'
                               else norm_w) / max(duration, 1))

    return {
        'overall_class'   : overall_class,
        'overall_conf'    : overall_conf,
        'is_shoplifting'  : overall_class == 'shoplifting',
        'behavior_events' : behavior_events,
        'person_summaries': {tid: t.to_dict()
                              for tid, t in person_trackers.items()},
        'attention_maps'  : dict(person_attn_maps),   # XAI output
        'duration'        : duration,
        'fps'             : fps,
        'total_frames'    : frame_num,
    }

print('run_inference() ready  (CNN-BiLSTM-Attention).')
print("Usage: result = run_inference('path/to/video.mp4')")
print("XAI  : result['attention_maps'] → {track_id: [[attn_weights], ...]}")


run_inference() ready  (CNN-BiLSTM-Attention).
Usage: result = run_inference('path/to/video.mp4')
XAI  : result['attention_maps'] → {track_id: [[attn_weights], ...]}


In [24]:
# ============================================================
# CELL 8 — Intent Scoring + Bias-Aware Assessment (XAI Layer)
# ============================================================
# Your thesis contribution: decomposed, interpretable scoring.
# Score = 50% behaviour + 30% product/concealment + 20% duration
# Bias adjustment: quality-based score correction.
# ============================================================
import numpy as np

def calculate_intent_score(behavior_events, video_duration,
                            person_summaries=None):
    """Compute 0-1 intent score. Returns score, severity, explanation."""
    shop_events = [e for e in behavior_events
                   if e['behavior_type'] == 'shoplifting']

    # Component 1 — behaviour (MobileNetV2 output)
    if shop_events:
        avg_conf = np.mean([e['confidence'] for e in shop_events])
        count_f  = min(1.0, len(shop_events) / 3.0)
        s_behav  = avg_conf * count_f
    else:
        s_behav  = 0.0

    # Component 2 — duration in suspicious state
    sus_secs   = sum(e['end_time'] - e['start_time'] for e in shop_events)
    s_duration = min(1.0, (sus_secs / max(1.0, video_duration)) * 3.0)

    # Component 3 — product / concealment evidence (YOLO)
    s_product     = 0.0
    product_flags = []
    if person_summaries:
        for tid, p in person_summaries.items():
            if p['left_with_goods']:
                s_product = max(s_product, 0.8)
                product_flags.append(
                    f'Person {tid}: left store with goods (no checkout visit)')
            if p['concealment_ratio'] > 0.3:
                s_product = max(s_product, p['concealment_ratio'])
                product_flags.append(
                    f'Person {tid}: bag/backpack {p["concealment_ratio"]:.0%} of time')
            if p['max_products_held'] > 0:
                product_flags.append(
                    f'Person {tid}: max {p["max_products_held"]} product(s) detected')

    total = max(0.0, min(1.0,
        0.50 * s_behav + 0.30 * s_product + 0.20 * s_duration))

    if   total < THRESHOLD_LOW:      severity = 'NONE'
    elif total < THRESHOLD_MEDIUM:   severity = 'LOW'
    elif total < THRESHOLD_HIGH:     severity = 'MEDIUM'
    elif total < THRESHOLD_CRITICAL: severity = 'HIGH'
    else:                             severity = 'CRITICAL'

    explanation = []
    if shop_events:
        explanation.append(
            f'{len(shop_events)} shoplifting segment(s) ({sus_secs:.1f}s)')
    explanation.extend(product_flags)
    if not explanation:
        explanation.append('No suspicious activity detected')

    return {
        'score'      : total,
        'severity'   : severity,
        'explanation': '; '.join(explanation) + f'. Score: {total:.2f} ({severity})',
        'components' : {'behaviour': s_behav,
                         'product'  : s_product,
                         'duration' : s_duration},
    }


def bias_aware_adjustment(intent_dict, quality_score=1.0):
    """
    Quality-based score correction (NOT demographic bias).
    Reduces score when video quality is low — prevents false positives
    from poor lighting / occlusion. See thesis Section 6.x.
    """
    raw   = intent_dict['score']
    flags = []
    adj   = 1.0
    if quality_score < 0.5:
        adj = min(adj, 0.70)
        flags.append('Low video quality — score reduced')
    elif quality_score < 0.75:
        adj = min(adj, 0.85)
        flags.append('Moderate video quality — score slightly reduced')
    if raw >= 0.5:
        adj = adj ** 0.5   # soften adjustment at high scores
    adj_score = max(0.0, min(1.0, raw * adj))
    return {
        'raw_score'      : raw,
        'adjusted_score' : adj_score,
        'fairness_score' : max(0.0, quality_score * (1.0 - 0.2 * len(flags))),
        'adj_factor'     : adj,
        'flags'          : flags,
        'requires_review': len(flags) > 0 or raw > THRESHOLD_HIGH,
    }


def generate_alert(intent_dict, bias_result, behavior_events,
                   pos_results=None):
    """Generate alert dict if score exceeds THRESHOLD_MEDIUM."""
    score    = bias_result['adjusted_score']
    severity = intent_dict['severity']
    if score < THRESHOLD_MEDIUM:
        return None

    n_sus    = sum(1 for e in behavior_events
                   if e['behavior_type'] == 'shoplifting')
    alert_id = f"ALERT-{datetime.now().strftime('%Y%m%d%H%M%S')}"

    pos_note = ''
    if pos_results:
        mismatches = [r for r in
                      (pos_results.values() if isinstance(pos_results, dict)
                       else pos_results)
                      if isinstance(r, dict) and r.get('mismatch') is True]
        if mismatches:
            pos_note = (f' POS MISMATCH: {len(mismatches)} person(s) '
                        f'with unaccounted items.')

    return {
        'alert_id'             : alert_id,
        'timestamp'            : datetime.now().isoformat(),
        'level'                : severity,
        'score'                : score,
        'message'              : (f'{n_sus} suspicious segment(s). '
                                   f'Risk: {score:.2f} ({severity}).{pos_note} '
                                   f'Human review required.'),
        'requires_human_review': True,
        'fairness_score'       : bias_result['fairness_score'],
    }


print('Intent scoring + bias assessment ready.')


Intent scoring + bias assessment ready.


In [25]:
# ============================================================
# CELL 9 — End-to-End Analysis + Case File
# ============================================================
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

def plot_behavior_timeline(behavior_events, duration,
                            title='Behaviour Timeline'):
    colors = {'normal': '#2ecc71', 'shoplifting': '#e74c3c'}
    fig, ax = plt.subplots(figsize=(14, 3))
    for e in behavior_events:
        ax.barh(0,
                max(0.1, e['end_time'] - e['start_time']),
                left=e['start_time'], height=0.6,
                color=colors.get(e['behavior_type'], '#95a5a6'),
                alpha=0.8, edgecolor='white')
    patches = [mpatches.Patch(color=c, label=k)
               for k, c in colors.items()
               if any(e['behavior_type'] == k for e in behavior_events)]
    ax.legend(handles=patches, loc='upper right', fontsize=9)
    ax.set_xlim(0, max(duration, 1))
    ax.set_xlabel('Time (seconds)'); ax.set_yticks([])
    ax.set_title(title, fontweight='bold')
    plt.tight_layout(); plt.show()


def analyze_video(video_path, quality_score=1.0, frame_step=1):
    """
    Basic end-to-end analysis (no POS prompt).
    Use analyze_video_with_pos() in Cell 10 for full POS verification.
    """
    print(f'Analyzing: {video_path}')
    inf    = run_inference(video_path, frame_step=frame_step)
    intent = calculate_intent_score(
        inf['behavior_events'], inf['duration'], inf['person_summaries'])
    bias   = bias_aware_adjustment(intent, quality_score)
    alert  = generate_alert(intent, bias, inf['behavior_events'])

    sep = '=' * 65
    print(sep)
    print('  DIGITAL WITNESS — ANALYSIS RESULTS')
    print(sep)
    print(f'  Classification : {inf["overall_class"].upper()}')
    print(f'  Confidence     : {inf["overall_conf"]:.1%}')
    print(f'  Intent Score   : {intent["score"]:.3f}  [{intent["severity"]}]')
    print(f'  Adjusted Score : {bias["adjusted_score"]:.3f}')
    print(f'  Fairness Score : {bias["fairness_score"]:.1%}')
    print(f'  Explanation    : {intent["explanation"]}')
    print()
    print('  Product summary (per tracked person):')
    for tid, p in inf['person_summaries'].items():
        print(f'    Person {tid}: products={p["max_products_held"]}  '
              f'concealment={p["concealment_ratio"]:.0%}  '
              f'checkout={p["checkout_visited"]}  '
              f'left_with_goods={p["left_with_goods"]}')
    if alert:
        print()
        print(f'  *** ALERT [{alert["level"]}] {alert["alert_id"]} ***')
        print(f'  {alert["message"]}')
    print()
    print('  NOTE: Advisory only — human review required.')
    print(sep)

    case_id   = f"CASE-{datetime.now().strftime('%Y%m%d%H%M%S')}"
    case_data = {
        'case_id'         : case_id,
        'timestamp'       : datetime.now().isoformat(),
        'system'          : 'Digital Witness — YOLO26n + MobileNetV2',
        'video'           : {'path': str(video_path),
                              'duration': inf['duration'],
                              'fps': inf['fps'],
                              'frames': inf['total_frames']},
        'classification'  : {'label': inf['overall_class'],
                              'confidence': inf['overall_conf'],
                              'is_shoplifting': inf['is_shoplifting']},
        'behavior_events' : inf['behavior_events'],
        'person_summaries': inf['person_summaries'],
        'intent_score'    : intent,
        'bias_assessment' : bias,
        'alert'           : alert,
        'advisory_note'   : 'Advisory system only. Does not determine guilt.',
    }
    case_path = CASE_DIR / f'{case_id}.json'
    with open(case_path, 'w') as f:
        json.dump(case_data, f, indent=2)

    if inf['behavior_events']:
        plot_behavior_timeline(
            inf['behavior_events'], inf['duration'],
            title=f'Behaviour Timeline — {Path(video_path).name}')

    print(f'Case file: {case_path}')
    return case_id, str(case_path), case_data


print('analyze_video() ready.')
print()
print("Basic  : case_id, path, r = analyze_video('video.mp4')")
print("Fast   : case_id, path, r = analyze_video('video.mp4', frame_step=3)")


analyze_video() ready.

Basic  : case_id, path, r = analyze_video('video.mp4')
Fast   : case_id, path, r = analyze_video('video.mp4', frame_step=3)


In [26]:
# ============================================================
# CELL 10 — Full Pipeline with POS Operator Verification
# ============================================================
# Runs inference then prompts the operator interactively:
#   1. Shows what YOLO detected for each person
#   2. Finds matching POS transaction by timestamp
#   3. Operator confirms or overrides the billed count
#   4. Flags any mismatch and saves POS report
# ============================================================

def analyze_video_with_pos(video_path, quality_score=1.0,
                             frame_step=1, video_timestamp=None,
                             tolerance_seconds=120):
    """
    Full pipeline with interactive POS verification.

    Args:
        video_path        : path to video
        quality_score     : 0-1 (1.0 = good quality)
        frame_step        : 1=every frame, 3=faster
        video_timestamp   : datetime the video was recorded
                            (matched to POS by time — defaults to now)
        tolerance_seconds : max seconds between video and POS timestamp
    """
    if video_timestamp is None:
        video_timestamp = datetime.now()

    print(f'Analyzing      : {video_path}')
    print(f'Video timestamp: {video_timestamp.strftime("%Y-%m-%d %H:%M:%S")}')
    print()

    # 1. Inference
    inf = run_inference(video_path, frame_step=frame_step)

    # 2. Intent scoring
    intent = calculate_intent_score(
        inf['behavior_events'], inf['duration'], inf['person_summaries'])

    # 3. Bias adjustment
    bias = bias_aware_adjustment(intent, quality_score)

    # 4. Operator POS verification (interactive prompt)
    print('--- Starting Operator Verification ---')
    print()
    pos_results = prompt_operator_verification(
        inference_result  = inf,
        pos_db            = pos_db,
        video_timestamp   = video_timestamp,
        tolerance_seconds = tolerance_seconds,
    )

    # 5. Alert (includes POS mismatch signals)
    alert = generate_alert(
        intent, bias, inf['behavior_events'],
        {i: r for i, r in enumerate(pos_results)})

    # 6. Case file
    case_id   = f"CASE-{datetime.now().strftime('%Y%m%d%H%M%S')}"
    case_data = {
        'case_id'         : case_id,
        'timestamp'       : datetime.now().isoformat(),
        'system'          : 'Digital Witness — YOLO26n + MobileNetV2 + POS',
        'video'           : {'path': str(video_path),
                              'duration': inf['duration'],
                              'fps': inf['fps'],
                              'frames': inf['total_frames']},
        'classification'  : {'label': inf['overall_class'],
                              'confidence': inf['overall_conf'],
                              'is_shoplifting': inf['is_shoplifting']},
        'behavior_events' : inf['behavior_events'],
        'person_summaries': inf['person_summaries'],
        'intent_score'    : intent,
        'bias_assessment' : bias,
        'alert'           : alert,
        'advisory_note'   : 'Advisory system only. Does not determine guilt.',
    }
    case_path = CASE_DIR / f'{case_id}.json'
    with open(case_path, 'w') as f:
        json.dump(case_data, f, indent=2)

    # 7. POS integrity report
    pos_report, pos_report_path = generate_pos_report(
        inf, pos_results, case_id, OUTPUTS_DIR)

    # 8. Timeline plot
    if inf['behavior_events']:
        plot_behavior_timeline(
            inf['behavior_events'], inf['duration'],
            title=f'Behaviour Timeline — {Path(video_path).name}')

    print(f'\nCase file  : {case_path}')
    print(f'POS report : {pos_report_path}')
    return case_id, case_data, pos_report


print('analyze_video_with_pos() ready.')
print()
print('Usage:')
print('  # Basic (timestamp = now, matches the suspicious test session):')
print("  case_id, r, pos = analyze_video_with_pos('path/to/video.mp4')")
print()
print('  # When prompted, enter the billed count for each person.')
print('  # Try entering a number different from detected to see the mismatch flag.')
print()
print('  # Faster (every 3rd frame):')
print("  case_id, r, pos = analyze_video_with_pos('video.mp4', frame_step=3)")


analyze_video_with_pos() ready.

Usage:
  # Basic (timestamp = now, matches the suspicious test session):
  case_id, r, pos = analyze_video_with_pos('path/to/video.mp4')

  # When prompted, enter the billed count for each person.
  # Try entering a number different from detected to see the mismatch flag.

  # Faster (every 3rd frame):
  case_id, r, pos = analyze_video_with_pos('video.mp4', frame_step=3)
